In [ ]:
# %%
# =================================================================================
# Step 1: Install and Import Necessary Libraries
# =================================================================================
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset, DataLoader
import gc

from tqdm import tqdm
import matplotlib.pyplot as plt
import pandas as pd
import csv
import pickle
import numpy as np
import random
from PIL import Image
from torch.nn import functional as F
import torchvision.transforms.functional as TF
import sys

In [ ]:
!pip uninstall -y timm
sys.path.append("/kaggle/input/pytorch-image-models")
import timm
# timm.list_models("vit_*_dinov2")

In [ ]:
# =================================================================================
# Step 2: Configuration
# =================================================================================

# --- Model & Training Settings ---
MODEL_NAME = 'vit_small_patch14_dinov2'
# CIFAR-100 has 100 classes
NUM_CLASSES = 100
# Adjust based on your GPU memory
BATCH_SIZE = 1000
# ViT models have a fixed input size
IMG_SIZE = 32
LEARNING_RATE = 5e-3
# Number of training epochs
EPOCHS = 150
HAS_POS = False
OVERLAP = 2
pretrained = None
WANDB = True
SEED = 56
WDECAY = 0.1
hid = 3
VAL_STEPS = 500
ALPHA = 3.0
Use_Patch_Position_Loss = False
Use_Row_Col_Loss = True
RC_ALPHA = 30.0

# Path to the CIFAR-100 dataset on Kaggle
BASE_PATH = '/kaggle/input/cifar100'

# --- Device Configuration ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {DEVICE}")

In [ ]:
torch.backends.cudnn.deterministic=True
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
if WANDB:
    import wandb
    # from wandb.keras import WandbCallback
    wandb.login(key='bb050692d5a8ea8b20a38ddcd72a9eb06f497aff')

In [ ]:
def unpickle(file):
    """
    A function to load the CIFAR-100 data from a pickled file.
    """
    with open(file, 'rb') as fo:
        # The 'latin1' encoding is required for compatibility with the original dataset files.
        dict = pickle.load(fo, encoding='latin1')
    return dict

# --- Step 1: Load the data from the files ---
# Adjust these paths to where you have saved the dataset. In a Kaggle environment,
# this path is typically '/kaggle/input/cifar100/'.
try:
    train_dict = unpickle('/kaggle/input/cifar100/train')
    test_dict = unpickle('/kaggle/input/cifar100/test')
    meta_dict = unpickle('/kaggle/input/cifar100/meta')
except FileNotFoundError:
    print("Please adjust the file paths to point to your local CIFAR-100 directory.")
    # Use dummy dicts to allow the rest of the code to be checked
    train_dict = {'data': np.zeros((1, 3072)), 'fine_labels': [0]}
    test_dict = {'data': np.zeros((1, 3072)), 'fine_labels': [0]}

In [ ]:
from PIL import Image
# --- Step 2: Define a Custom Dataset Class ---
class CustomCIFAR100(Dataset):
    def __init__(self, data_dict, transform=None):
        self.data = data_dict['data'].reshape(-1, 3, 32, 32)
        # Transpose from (N, 3, 32, 32) to (N, 32, 32, 3) for PIL
        self.data = self.data.transpose((0, 2, 3, 1))
        self.labels = data_dict['fine_labels']
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Get image and label at the index
        image = self.data[idx]
        label = self.labels[idx]

        # Convert numpy array to PIL Image
        image = Image.fromarray(image)

        # ✨ Apply transforms here!
        if self.transform:
            image = self.transform(image)

        return image, label

# --- Step 3: Define Your Transformations ---
# Now you can include data augmentation
train_transforms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

# --- Step 4: Create Datasets and DataLoaders ---
train_dataset = CustomCIFAR100(data_dict=train_dict, transform=train_transforms)
test_dataset = CustomCIFAR100(data_dict=test_dict, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Now it works as expected!
print("Successfully created DataLoaders with transforms.")
images, labels = next(iter(train_loader))
print(f"Batch of images shape: {images.shape}")

In [ ]:
# %% [code]
# =================================================================================
# Step 3.5: Visualize a Batch of Training Data
# =================================================================================
import matplotlib.pyplot as plt
import numpy as np
import torchvision

img_mean, img_std = (0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)
def imshow(inp, title=None):
    """A helper function to denormalize and display an image tensor."""
    # Define the same mean and std used for normalization
    mean = np.array(img_mean)
    std = np.array(img_std)
    
    # Transpose from (C, H, W) to (H, W, C)
    inp = inp.numpy().transpose((1, 2, 0))
    # Denormalize
    inp = std * inp + mean
    # Clip values to be between 0 and 1
    inp = np.clip(inp, 0, 1)
    
    plt.imshow(inp)
    if title is not None:
        plt.title(title, fontsize=10)
    plt.axis('off')

# Get one batch of training images
try:
    inputs, classes = next(iter(train_loader))
    
    # Get the class names from the dataset object
    class_names = meta_dict['fine_label_names']

    # Create a grid of images
    fig = plt.figure(figsize=(16, 8))
    plt.suptitle("Sample Images from CIFAR-100 Dataset", fontsize=16)
    
    # Display the first 16 images from the batch
    for i in range(16):
        ax = plt.subplot(4, 8, i + 1)
        class_name = class_names[classes[i]]
        imshow(inputs[i], title=class_name)
        
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

except NameError as e:
    print(e, "Could not display images. Please ensure the previous cells have been run to create 'train_loader'.")


In [ ]:
if WANDB:        
    params = {}
    params['NUM_CLASSES'] = NUM_CLASSES
    params['BATCH_SIZE'] = BATCH_SIZE
    params['IMG_SIZE'] = IMG_SIZE
    params['EPOCHS'] = EPOCHS
    # params['START_EPOCH'] = START_EPOCH
    # params["offset"] = offset
    params['HAS_POS'] = HAS_POS
    params['OVERLAP'] = OVERLAP
    params["ALPHA"] = ALPHA
    params["Row_Col_Loss"] = Use_Row_Col_Loss
    params["RC_ALPHA"] = RC_ALPHA
    params['lr'] = LEARNING_RATE
    params['train_imgs'] = len(train_dataset)
    params['hid'] = hid
    params['seed'] = SEED
    wandb.init(
#             reinit=True,
        # set the wandb project where this run will be logged
        project="dinov2_pos_cifar100",
        # track hyperparameters and run metadata
        config=params,
        group='cifar100',
        job_type='val'
    )

In [ ]:
# MODEL_NAME = "vit_rope_small_patch14_dinov2"
# =================================================================================
# Step 4: Initialize the Model, Loss Function, and Optimizer
# =================================================================================
# --- Model ---
print(f"🤖 Initializing model: {MODEL_NAME} for {NUM_CLASSES} classes...")
model = timm.create_model(
    MODEL_NAME,
    pretrained=False, # As requested: trains the model from scratch
    patch_size=4,
    num_classes=NUM_CLASSES, # Set the classifier head to 100 classes
    img_size=IMG_SIZE,
).to(DEVICE)

# feature_layers = [2, 5, 8, 11]
# dummy_input = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
# with torch.no_grad():
#     feats = model.forward_features(dummy_input)
#     multi_feats = model.forward_intermediates(dummy_input, indices=feature_layers, intermediates_only=True)


# print(f"Model created successfully!")
# print(f"Input shape: {dummy_input.shape}")
# print(f"Output shape: {feats.shape}") 
# print(f"multi_feats shape: {multi_feats[-1].shape} X {len(multi_feats)}")
# del feats, multi_feats, dummy_input
# gc.collect()

In [ ]:
print('model.patch_embed.proj', model.patch_embed.proj)
if OVERLAP>0:
    # Customize patch embedding for overlap (e.g., patch_size=15, stride=14)
    original_patch_size = model.patch_embed.proj.kernel_size[0]
    new_patch_size = original_patch_size + OVERLAP  # Or 15, 16, 17, etc., as desired
    stride = original_patch_size
    original_grid_size = IMG_SIZE // stride  # 16 for 224//14
    padding = ((original_grid_size - 1) * stride + new_patch_size - IMG_SIZE + 1) // 2  # +1 for ceiling effect; yields 1 for patch_size=15
    
    # Override the PatchEmbed projection (Conv2d layer)
    in_chans = model.patch_embed.proj.in_channels  # Typically 3 for RGB
    embed_dim = model.patch_embed.proj.out_channels  # e.g., 768 for base
    model.patch_embed.proj = nn.Conv2d(
        in_chans, embed_dim,
        kernel_size=(new_patch_size, new_patch_size),
        stride=(stride, stride),
        padding=padding  # Updated to ensure full coverage and original grid size
    ).to(DEVICE)
    
    # Recompute grid size and num_patches
    # grid_size_h = ((IMG_SIZE + 2 * padding - new_patch_size) // stride) + 1
    # grid_size_w = grid_size_h  # Assuming square input
    # print(new_patch_size, padding, grid_size_h, model.patch_embed.grid_size)
    # model.patch_embed.grid_size = (grid_size_h, grid_size_w)
    # model.patch_embed.num_patches = grid_size_h * grid_size_w
    # print(f"Updated to patch_size={new_patch_size}, stride={stride}, padding={padding}, num_patches={model.patch_embed.num_patches}")


if not HAS_POS and hasattr(model, 'pos_embed') and model.pos_embed is not None:
    model.pos_embed.data.zero_()
    model.pos_embed.requires_grad = False
    print("✅ Positional embedding has been disabled.")
if pretrained:
    state_dicts = torch.load(pretrained, map_location=DEVICE)
    IncompatibleKeys = model.load_state_dict(state_dicts)
    print(IncompatibleKeys)
# --- Loss Function & Optimizer ---
criterion = nn.CrossEntropyLoss()
# optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WDECAY)
print("✅ Model, Loss Function, and Optimizer are ready.")

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
print("✅ Model, Loss, Optimizer, and LR Scheduler are ready.")

In [ ]:
class PatchRowColCriterion(nn.Module):
    def __init__(self, feat_dim, grid_h, grid_w):
        """
        Predict row and column of each patch independently.

        Args:
            feat_dim (int): Dimension of patch features (D)
            grid_h (int): Number of patch rows
            grid_w (int): Number of patch columns
        """
        super().__init__()
        self.grid_h = grid_h
        self.grid_w = grid_w

        # MLP for row prediction
        self.row_mlp = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Linear(256, grid_h)
        )

        # MLP for column prediction
        self.col_mlp = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Linear(256, grid_w)
        )

        self.ce = nn.CrossEntropyLoss()

        # Precompute row/col labels
        rows = torch.arange(grid_h).unsqueeze(1).repeat(1, grid_w).flatten()
        cols = torch.arange(grid_w).repeat(grid_h)
        self.register_buffer("row_labels", rows)
        self.register_buffer("col_labels", cols)

    def forward(self, feats):
        """
        Args:
            feats: (B, N, D) patch features, N = grid_h * grid_w
        Returns:
            avg_loss: scalar, sum of row and column classification losses
        """
        B, N, D = feats.shape
        assert N == self.grid_h * self.grid_w, f"Expected {self.grid_h*self.grid_w} patches, got {N}"

        x = feats.reshape(-1, D)  # (B*N, D)

        # Repeat labels for batch
        row_labels = self.row_labels.repeat(B)
        col_labels = self.col_labels.repeat(B)

        # Predict rows and columns
        row_logits = self.row_mlp(x)
        col_logits = self.col_mlp(x)

        # Compute cross-entropy loss for rows and columns
        loss_row = self.ce(row_logits, row_labels)
        loss_col = self.ce(col_logits, col_labels)

        return (loss_row + loss_col) / 2  # average


In [ ]:
if Use_Row_Col_Loss:
    grid_h, grid_w = model.patch_embed.grid_size
    rowcol_loss = PatchRowColCriterion(
        feat_dim=model.get_classifier().in_features,
        grid_h=grid_h,
        grid_w=grid_w
    ).to(DEVICE)

In [ ]:
import csv

# FP16: Initialize the Gradient Scaler
scaler = torch.amp.GradScaler('cuda')
# =================================================================================
# Step 5: Training and Validation Loop
# =================================================================================
print(f"\n🚀 Starting training for {MODEL_NAME}...")

# ✅ Initialize training_history as a dictionary of lists
training_history = {
    'train_loss': [],
    'train_acc': [],
    'valid_acc': [],
    'epoch': [],
}

for epoch in range(EPOCHS):
    # --- Training Phase ---
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Training]")
    
    # FP16: Use autocast for the forward pass
    for inputs, labels in train_pbar:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            feats = model.forward_features(inputs)
            outputs = model.forward_head(feats)
            # outputs = model(inputs)
            loss = criterion(outputs, labels)

            if Use_Row_Col_Loss:
                aux_loss = rowcol_loss(feats[:, 1:, :])
                # print(loss, aux_loss)
                loss = loss + RC_ALPHA * aux_loss
        
        # FP16: Scale, backward, and step
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
        
        batch_acc = (predicted == labels).sum().item() / labels.size(0)
        train_pbar.set_postfix({'loss': loss.item(), 'acc': f'{batch_acc:.2f}'})

    epoch_train_loss = running_loss / len(train_loader.dataset)
    epoch_train_acc = train_correct / train_total

    # if (epoch + 1) % 2 == 0:
    # --- Validation Phase ---
    model.eval()
    val_correct = 0
    val_total = 0
    val_pbar = tqdm(test_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]")
    
    with torch.no_grad():
        for inputs, labels in val_pbar:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    epoch_val_acc = val_correct / val_total
    
    print(f"\nEpoch {epoch+1}/{EPOCHS} Summary:")
    print(f"  Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | Valid Acc: {epoch_val_acc:.4f}\n")

    # ✅ Append the results to the correct lists within the dictionary
    training_history['train_loss'].append(epoch_train_loss)
    training_history['train_acc'].append(epoch_train_acc)
    training_history['valid_acc'].append(epoch_val_acc) 
    training_history['epoch'].append(epoch+1)

    # Update the learning rate scheduler
    if 'scheduler' in locals():
        scheduler.step()

print("🏁 Training complete.")

# =================================================================================
# Step 6: Save the Results and Model
# =================================================================================

# ✅ Step 1: Convert the dictionary directly into a pandas DataFrame
history_df = pd.DataFrame(training_history)

# ✅ Step 2: Add the 'epoch' column at the beginning
# Create the list of epochs where validation was actually performed
# epochs_validated = range(5, EPOCHS + 1, 5) 
# history_df.insert(0, 'epoch', epochs_validated)

# ✅ Step 3: Save the DataFrame to a CSV file
csv_file = 'training_history.csv'
history_df.to_csv(csv_file, index=False) # index=False prevents pandas from writing row numbe

print(f"✅ Training history saved to '{csv_file}'")

# Save the model's state dictionary
torch.save(model.state_dict(), f'{MODEL_NAME}_final.pth')
print(f"✅ Model saved to '{MODEL_NAME}_final.pth'")

In [ ]:
if WANDB:
    best_index = max(range(len(training_history['valid_acc'])), key=lambda i: training_history['valid_acc'][i])
    best_accuracy = training_history['valid_acc'][best_index]
    best_epoch = training_history['epoch'][best_index]
    # best_step = training_history['step'][best_index]
    # best_accuracy = max(training_history['valid_acc'])
    # best_index = training_history['valid_acc'].index(best_accuracy)
    # best_epoch = (best_index + 1) * VAL_STEPS + START_EPOCH * steps_per_epoch
    train_accuracy = max(training_history['train_acc'])
    wandb.log({"best_acc": best_accuracy, "best_epoch": best_epoch, "train_acc": train_accuracy})

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# try:
#     # Use pandas to read the CSV file into a DataFrame
#     # history_df = pd.read_csv(csv_file)
    
#     # print(f"✅ Successfully loaded data from '{csv_file}':")

#     # # Convert the list of dictionaries to a pandas DataFrame for easy plotting
#     history_df = pd.DataFrame(training_history)    

# except FileNotFoundError:
#     history_df = None
#     print(f"❌ Error: The file '{csv_file}' was not found.")
#     print("Please make sure you have run the training loop to save the file first.")
    
# First, ensure the training_history list is not empty
if history_df is None:
    print("Training history is empty. Please run the training loop first.")
else:
    # --- Create a single figure and axis for the plot ---
    fig, ax = plt.subplots(figsize=(12, 7))
    plt.title('Training and Validation Accuracy Over Epochs', fontsize=16)
    
    # --- Plot Training & Validation Accuracy ---
    ax.plot(history_df['epoch'], history_df['train_acc'], 's--', color='tab:green', label='Training Accuracy')
    ax.plot(history_df['epoch'], history_df['valid_acc'], '^-', color='tab:blue', label='Validation Accuracy')
    
    # --- Set labels and legend ---
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    
    # Set the y-axis to be formatted as percentages
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.set_ylim(0, 1) # Set y-axis limits from 0 to 1 for accuracy

    # Set the x-axis to show integer epoch numbers
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

    plt.tight_layout()
    plt.show()

In [ ]:
if WANDB:
    wandb.finish()